# Prepare the contend based on one model

This notebook contains the steps to generate the SSOT from an ODM model and render output channels.

This notebook will be executed by the run.bat

In [ ]:
odm_source_folder = 'IM'
destination_folder = 'content'

# Skip generation of the SSOT from ODM? 
skip_ssot_generation = False

# Location of the python tools
tools_path = 'pythonWork/pythonSource'

In [ ]:
notebook_version = "0.6"

In [ ]:
import sys
import os
from pathlib import Path
import glob
import json

In [ ]:
import logging
from logging import handlers

os.makedirs('log', exist_ok=True)
logfile = 'log/generator.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024*1024*10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

In [ ]:
sys.argv[0]

In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Generate SSOT and diagrams from ODM model')
parser.add_argument('--model', '-m', dest='source_folder', default=odm_source_folder)
parser.add_argument('--destination', dest='destination_folder', default=destination_folder)
parser.add_argument('--skip-ssot', '-f', action='store_true', dest='skip_odm', help="Skip generation of SSOT out of Oracle Database Modeler model")
parser.add_argument('--skip-web', '-w', action='store_true', dest='skip_web', help="Skip generation of static web content")
parser.add_argument('--confluence', '-c', action='store_true', dest='confluence', help="Skip generation of Confluence content")
parser.add_argument('--sharepoint', '-s', action='store_true', dest='sharepoint', help="Enable generation of Sharepoint content")
parser.add_argument('--dont-merge', action="store_true", dest="clean_slate", help="Do not merge! Existing database is renamed to .bkp")
parser.add_argument('--version', action='store_true')

arguments = argparse.Namespace()

if len(sys.argv) > 0 and '.py' in sys.argv[0] and not 'ipykernel' in sys.argv[0]:
    arguments = parser.parse_args()
    odm_source_folder = arguments.source_folder
    destination_folder = arguments.destination_folder
    skip_ssot_generation = arguments.skip_odm
    if arguments.version:
        print(f"Version {notebook_version}")
        exit(0)
else:
    # in jupyter environment
    assert 'ipykernel' in sys.argv[0], f"Not running in Jupyter environment 🙀"
    odm_source_folder = '../../testdata/fyyccim-refmodels/raetsel-3lang/IM'
    destination_folder = 'content'
    tools_path = os.path.abspath('../../pythonWork/pythonSource')
    skip_ssot_generation = False
    arguments = argparse.Namespace( **{ 'clean_slate': True } )


In [ ]:
arguments

In [ ]:
logger.info(f"Starting generator version {notebook_version}")

### Safeguards


In [ ]:
models = glob.glob(odm_source_folder + '/*.[dD][mM][dD]')
if len(models) < 1:
    logger.fatal(f"No model file (*.dmd) found in source folder {os.path.abspath(odm_source_folder)}.")
    exit(2)

model = models[0]
if len(models) > 1:
    for name in models:
        # pick model with shortest name
        if len(name) < len(model):
            model = name
    logger.warning(f"Found {len(models)} models in {os.path.abspath(odm_source_folder)} using {model}")
    # disable other models?
    for name in models:
        if name != model:
            os.rename(name, os.path.splitext(name)[0] + '.hidden')

In [ ]:
base_path = Path(odm_source_folder)
assert os.path.isdir(odm_source_folder), "Cannot find source folder {}".format(odm_source_folder)
project_files = list(base_path.glob('*.dmd'))
assert len(project_files) == 1, "Cannot find exactly 1 ODM .dmd file in source folder {}: {}".format(odm_source_folder, project_files)
logger.info(f"Processing the information model in {os.path.abspath(base_path)}")
from IPython.core.display import HTML
HTML('<span style="font-family: Impact; font-size:48px">Processing the information model in<br/><span style="color: darkorange">{0}</span></span>'.format(os.path.abspath(odm_source_folder)))

In [ ]:
if not (os.path.exists(tools_path)
        and os.path.isfile(os.path.join(tools_path, 'IM_ODM', 'transferModel.py'))):
    logging.fatal(f"Tools not in expected path {tools_path}")
    exit(3)

logger.debug(f"Working with tools in {tools_path}")

# Add toolbox to python library path
sys.path.insert(0, os.path.abspath(tools_path))

# HACK around issue #xxx
sys.path.insert(1, os.path.abspath(os.path.join(tools_path, 'IM_db')))
sys.path.insert(1, os.path.abspath(os.path.join(tools_path, 'IM_WEB')))

from SSOT_infra import parameters
from IM_db.IM_JSON import JSModel
from IM_WEB.IM_HTML import entityenviron
from IM_WEB.IM_HTML import drawiodiagram

In [ ]:
from SSOT_infra import logmessages

def tap_logmessages(message: str):
    logger.warning(message)

logmessages.logtrap = tap_logmessages

# Process the datasource

In [ ]:
global parameter

parameters.initparam(odm_source_folder)

config_folder_before = None

odm_config_folder = os.path.join(odm_source_folder, parameters.odmKonfDirec())
if not os.path.isdir(odm_config_folder):
#    parameters.parameter['odmkonfdirec'] = 'Configuration/'
    odm_config_folder = os.path.join(odm_source_folder, parameters.odmKonfDirec())
    logger.warning(f"Patching config folder to {parameters.odmKonfDirec()}")
    config_folder_before = os.path.join(odm_source_folder, 'Configuration')
    assert os.path.isdir(config_folder_before), f"Missing configuration folder {config_folder_before}"
    os.rename(config_folder_before, odm_config_folder)

assert os.path.isdir(odm_config_folder), f"Configuration folder {os.path.abspath(odm_config_folder)} not found"
assert parameters.odmKonfDirec()[-1] == '/'

In [ ]:
sqlfilepath = os.path.join(tools_path, 'IM_db/sqlfiles/')

In [ ]:
# Ensure base folder exists
base = os.path.split(parameters.dbFilePath())[0]
os.makedirs(base, exist_ok = True)

In [ ]:
if arguments.clean_slate and os.path.isfile(parameters.dbFilePath()):
    existing_database = Path(parameters.dbFilePath())
    logging.info(f"Don't merge existing SSOT {existing_database}. It is backed up as {existing_database.with_suffix('.bkp')}")
    existing_database.rename(existing_database.with_suffix('.bkp'))

In [ ]:
from IM_ODM import fillDB

if not skip_ssot_generation:
    logger.info('Updating database {db} from model {odm}'.format(db=parameters.dbFilePath(), odm=parameters.dbDirect()))
    try:
        #fillDB.filldbmain(odm_source_folder, createnewdb=True)
        fillDB.main(odm_source_folder)
        logger.info(f"Sucessfully updated {parameters.dbFilePath()}")
    except:
        print('Consult logfile {}'.format(parameters.logfilepath()))
        print(f"Remove the half-filled database from {parameters.dbFilePath()} to restart the process")
        raise

In [ ]:
database_file = os.path.abspath(parameters.dbFilePath())
json_file = os.path.splitext(database_file)[0] + '.json'
assert os.path.isfile(json_file), f"SSOT file {json_file} not found"
logger.info(f"Loading SSOT from {json_file}")

In [ ]:
with open(json_file) as f:
    data = json.load(f)
assert len(data) > 0, f'Config is empty :-(' 

In [ ]:
jsmodel = JSModel.readfromfile(pfilename=json_file)

In [ ]:
import gettext

class Translator:
    """Translate strings"""
    logger = logging.getLogger("Translator")
    
    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher', './locale', fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        self.logger = logging.getLogger("Translator " + language)
        
    def tr(self, element) -> str:
        if isinstance(element, dict):
            """If the value provided is a field containing translations, use them"""
            text = element.get(self.language)
            if text is None: #and len(element.values()) > 0:
                result = list(element.values())[0]
                if result is not None:
                    self.logger.warning(f"No translation for {text}. Falling back to {result} from {str(element)}")
                    text = result
            if not text:
                return ''
            # Strip leading translation marker
            #if '*de* ' in text:
            #    text = text.replace('*de* ', '', 1)
            return text

        # Fallback to gettext if not a dict
        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning("Cannot translate element '{0}' of type {1}".format(element, type(element)))
        return None
    
    def gettext(self, text: str):
        result = self.translator.gettext(text)
        if result == text:
            self.logger.warning("No translation for {0}".format(text))
        return result
    
    def translator(self):
        return self.translator
    
    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title = title, language = self.language)
    
    def key_lang(self, key: str) -> str:
        return key + '-' + self.language
    
    def lang(self) -> str:
        return self.language

In [ ]:
translators = { language: Translator(language) for language in data['languages'] }
translators

In [ ]:
def sanitize_filename(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ('.', '-', '_', ' ')).rstrip()

# Render draw.io diagrams

In [ ]:
from lxml import etree

from tqdm.autonotebook import tqdm

content_root = '.'

diagram_count = len(data['diagrams'].keys())*len(translators)

path = os.path.join(destination_folder, 'diagrams')            
folder = os.path.join(content_root, path)
os.makedirs(folder, exist_ok=True)
logger.info(f"Rendering diagrams to {os.path.abspath(folder)}")

with tqdm(total=diagram_count, dynamic_ncols=True, unit='Diagram') as pbar:
    for lang, translator in translators.items():
        for key, diagram in data['diagrams'].items():

            filename = f"{key}-{sanitize_filename(diagram['name'])}-{lang}.drawio"
            file = os.path.join(folder, filename)
            
            pbar.set_description(f"Generating diagram {key} '{diagram['name']}' [{lang}] to {file}")
            draw_io_xml = drawiodiagram.create_diagram(key, jsmodel, translator)
            with open(file, 'wb') as out:
                out.write(etree.tostring(draw_io_xml))
                logger.debug(f"Diagram '{diagram['name']}' stored in draw.io format to {filename}")
            pbar.update(1)

In [ ]:
if config_folder_before is not None:
    logger.warning(f"Moving configuration folder back to {config_folder_before}")
    os.rename(odm_config_folder, config_folder_before)

In [ ]:
logger.info(f"Successfully created {diagram_count} diagrams to {os.path.join(destination_folder, 'diagrams')}")

# Create web content

In [ ]:
import listWebdoku
from IM_WEB.IM_HTML.printHTML import HTMLExport
from IM_OBJECTS import Languagetext

html_export = HTMLExport()
html_export.setmodel(jsmodel)

repo_root = os.getcwd()

if not os.path.isdir(os.path.join(repo_root, 'pythonWork')):
    repo_root = os.path.join(repo_root, '..', '..')
                   
repo_root = os.path.abspath(os.path.join(repo_root, 'pythonWork', 'pythonSource', 'IM_WEB', 'html-lib'))
html_export.setWebDirec(destination_folder)
logging.info(f"Web ressources {html_export.libSourceDirec}")
logging.info(f"Exporting HTML for language {Languagetext.reportLang()}")
listWebdoku.listwebmain(html_export, plang=None)